In [6]:
! python fine_tune.py

[hf] No HF token found. Downloads may be slower. Set HF_TOKEN for better reliability.
[data] Loading KisanVaani/agriculture-qa-english-only (train)
README.md: 1.74kB [00:00, 6.37MB/s]
data/train-00000-of-00001.parquet: 100% 1.97M/1.97M [00:01<00:00, 1.76MB/s]
Generating train split: 100% 22615/22615 [00:00<00:00, 348272.79 examples/s]
Map: 100% 22615/22615 [00:01<00:00, 20942.78 examples/s]
Filter: 100% 22615/22615 [00:00<00:00, 308064.15 examples/s]
[data] KisanVaani/agriculture-qa-english-only usable rows: 22574
Creating json from Arrow format: 100% 8/8 [00:00<00:00, 137.76ba/s]
[data] Wrote merged training JSONL: data/processed/agri_train.jsonl
[data] Total rows: 8000
config.json: 100% 659/659 [00:00<00:00, 4.39MB/s]
tokenizer_config.json: 7.30kB [00:00, 23.6MB/s]
vocab.json: 2.78MB [00:00, 72.7MB/s]
merges.txt: 1.67MB [00:00, 92.9MB/s]
tokenizer.json: 7.03MB [00:00, 152MB/s]
model.safetensors: 100% 988M/988M [00:04<00:00, 235MB/s]
Loading weights: 100% 290/290 [00:00<00:00, 718.57i

In [2]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load base model + LoRA adapter
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
adapter_path = "outputs/qwen_agri_lora"  # your --output-dir

tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

def ask(question: str, system_prompt: str = "You are an expert agriculture assistant. Give practical, concise, field-ready advice."):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
        )

    # Decode only the new tokens
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

# Test it
print(ask("What is the best fertilizer for wheat crops?"))
print(ask("How do I treat fungal infections in tomatoes?"))

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Wheat requires a higher amount of phosphorus and potassium than maize, which makes it an important crop for fertilizers. Phosphorus plays a crucial role in plant development and growth, while potassium helps improve root development and overall plant health. Fertilizers should be applied at appropriate rates based on soil analysis and crop requirements.
To effectively manage tomato fungal infections, it is recommended to apply fungicides or spray with appropriate substances at the specified intervals and concentrations based on the disease severity. Additionally, maintaining good plant hygiene by regularly cleaning plants and removing infected parts can help prevent infection.


In [4]:
ask("How much phosphorus per hectare should I apply to wheat?")


"A good amount of phosphorus for wheat depends on several factors such as the soil's phosphorus content, fertilization strategy, and the specific crop type. However, as a general guideline, typically around 10-20 kg/ha of phosphorus is recommended for wheat, which corresponds to about 30-60 ppm (parts per million) of phosphorus.\n\nIt's important to consult with local agricultural extension services or agronomists who can provide specific recommendations based on your location and farming conditions. They can also help determine the appropriate rate and frequency of phosphorus application for wheat in your area."

In [8]:
! python build_stage1_captions.py \
    --images_dir ./data/plantvillage \
    --output_jsonl ./data/captions/captions.jsonl \
    --shuffle \
    --max_images 20000

Traceback (most recent call last):
  File "/content/build_stage1_captions.py", line 180, in <module>
    main()
  File "/content/build_stage1_captions.py", line 142, in main
    raise FileNotFoundError(f"images_dir not found: {images_dir}")
FileNotFoundError: images_dir not found: data/plantvillage


In [6]:
! python train_projector.py \
    --stage 1 \
    --base_model Qwen/Qwen2.5-0.5B-Instruct \
    --lora_ckpt ./outputs/qwen_agri_lora \
    --data_dir ./data/captions \
    --output_dir ./checkpoints \
    --epochs 3 \
    --batch_size 4 \
    --lr 2e-4

INFO:__main__:Stage 1 training on cuda
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-large-patch14/32bd64288804d66eefd0ccbe215aa642df71cc41/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-large-patch14/32bd64288804d66eefd0ccbe215aa642df71cc41/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/model.safetensors "HTTP/1.1 302 Found"
Loading weights: 100% 391/391 [00:00<00:00, 431.77it/s, Materializing param=vision_model.pre_layrnorm.weight]
CLIPVisionModel LOAD REPORT from: openai/clip-vit-large-patch14
Key       

In [9]:
! mkdir -p ./data/captions ./data/vqa ./checkpoints


In [11]:
! python build_stage1_captions.py \
  --images_dir ./data/plantvillage \
  --output_jsonl ./data/captions/captions.jsonl \
  --shuffle \
  --max_images 20000


Traceback (most recent call last):
  File "/content/build_stage1_captions.py", line 198, in <module>
    main()
  File "/content/build_stage1_captions.py", line 154, in main
    raise FileNotFoundError(
FileNotFoundError: images_dir not found: data/plantvillage
Current working directory: /content
Detected subfolders under ./data: ['captions', 'processed', 'vqa']
Use --images_dir with the real dataset path (absolute or cwd-relative).


In [13]:
! mkdir -p /content/data

In [14]:
! git clone --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git /content/data/PlantVillage-Dataset

Cloning into '/content/data/PlantVillage-Dataset'...
remote: Enumerating objects: 163219, done.
remote: Counting objects: 100% (163219/163219), done.
remote: Compressing objects: 100% (163125/163125), done.
remote: Total 163219 (delta 93), reused 163214 (delta 93), pack-reused 0 (from 0)
Receiving objects: 100% (163219/163219), 2.00 GiB | 18.26 MiB/s, done.
Resolving deltas: 100% (93/93), done.
Updating files: 100% (182404/182404), done.


In [15]:
! python /content/build_stage1_captions.py --images_dir /content/data/PlantVillage-Dataset/raw/color --output_jsonl /content/data/captions/captions.jsonl --shuffle --max_images 20000

[done] Wrote 20000 caption samples to: /content/data/captions/captions.jsonl


In [16]:
! python /content/build_stage2_vqa.py --images_dir /content/data/PlantVillage-Dataset/raw/color --output_jsonl /content/data/vqa/vqa.jsonl --pairs_per_image 4 --shuffle --max_images 8000

[done] Wrote 32000 VQA samples to: /content/data/vqa/vqa.jsonl
[note] This dataset is template-generated. Manually review a sample before training.


In [ ]:
! python /content/train_projector.py --stage 1 --base_model Qwen/Qwen2.5-0.5B-Instruct --lora_ckpt /content/outputs/qwen_agri_lora --data_dir /content/data/captions --output_dir /content/checkpoints --epochs 3 --batch_size 4 --lr 2e-4

INFO:__main__:Stage 1 training on cuda
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-large-patch14/32bd64288804d66eefd0ccbe215aa642df71cc41/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-large-patch14/32bd64288804d66eefd0ccbe215aa642df71cc41/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/model.safetensors "HTTP/1.1 302 Found"
Loading weights: 100% 391/391 [00:00<00:00, 617.68it/s, Materializing param=vision_model.pre_layrnorm.weight]
CLIPVisionModel LOAD REPORT from: openai/clip-vit-large-patch14
Key       